# 02. Lower Bound, Upper Bound and "First True"

Most real binary search problems are not "find exactly x". They are "find the **first** position where something becomes true". This notebook gives you one template for all of them.

**Learning order:** WHY EXACT SEARCH IS NOT ENOUGH → LOWER BOUND → UPPER BOUND → FIRST AND LAST POSITION → UNIFIED TEMPLATE → CIRCULAR LETTERS → INTERVIEW THINKING

## 1. Problem with duplicates

In `[1, 2, 4, 4, 4, 7]`, searching for `4` may return index 2, 3 or 4. We often need the **first** or **last** one, or "where would it be inserted?".

## 2. Lower bound = first index with `nums[i] >= x`

Search space is `[lo, hi)` with `hi = len(nums)` (the answer can be `len(nums)` if everything is smaller).

- `nums[mid] < x` → mid is too small, answer is to the right → `lo = mid + 1`.
- otherwise `mid` **could be** the answer → keep it → `hi = mid` (not `mid - 1`).

Loop `while lo < hi`. When `lo == hi`, that is the answer.

In [1]:
def lower_bound(nums, x):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

a = [1, 2, 4, 4, 4, 7]
print(lower_bound(a, 4), lower_bound(a, 5), lower_bound(a, 8), lower_bound(a, 0))   # 2 5 6 0

2 5 6 0


## 3. Upper bound = first index with `nums[i] > x`

Only the condition changes: `nums[mid] <= x` → `lo = mid + 1`.

In [2]:
def upper_bound(nums, x):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo

print(upper_bound(a, 4))                       # 5
print(upper_bound(a, 4) - lower_bound(a, 4))   # 3  -> count of 4s

5
3


## 4. First and Last Position, Search Insert Position

- **Insert position** = `lower_bound`.
- **First position** of `x` = `lower_bound(x)` (check it really equals `x`).
- **Last position** of `x` = `upper_bound(x) - 1`.

In [3]:
def search_range(nums, target):
    first = lower_bound(nums, target)
    if first == len(nums) or nums[first] != target:
        return [-1, -1]
    return [first, upper_bound(nums, target) - 1]

print(search_range([5, 7, 7, 8, 8, 10], 8))   # [3, 4]
print(search_range([5, 7, 7, 8, 8, 10], 6))   # [-1, -1]

print([lower_bound([1, 3, 5, 6], t) for t in (5, 2, 7, 0)])   # [2, 1, 4, 0]  (Search Insert Position)

[3, 4]
[-1, -1]
[2, 1, 4, 0]


## 5. The unified template: "first True"

Many problems say: "there is a rule `pred(i)` that is `False False False True True True`. Find the first `True`."

```
lo, hi = start, end          # answer in [lo, hi]; hi is the "nothing is True" position
while lo < hi:
    mid = (lo + hi) // 2
    if pred(mid): hi = mid       # mid might be the answer, keep it
    else:         lo = mid + 1   # mid is False, discard it
return lo
```

**Why no infinite loop:** `mid < hi` always (floor), so `hi = mid` shrinks; `lo = mid + 1` shrinks.

In [4]:
def first_true(lo, hi, pred):
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

a = [1, 2, 4, 4, 4, 7]
print(first_true(0, len(a), lambda i: a[i] >= 4))   # 2  (= lower_bound)
print(first_true(0, len(a), lambda i: a[i] > 4))    # 5  (= upper_bound)

2
5


## 6. Find Smallest Letter Greater Than Target

Letters are sorted and **wrap around**. Answer = first letter `> target` = `upper_bound`. If it is past the end, wrap to index 0 with `% len`.

In [5]:
def next_greatest_letter(letters, target):
    idx = upper_bound(letters, target)
    return letters[idx % len(letters)]

print(next_greatest_letter(["c", "f", "j"], "a"))   # c
print(next_greatest_letter(["c", "f", "j"], "c"))   # f
print(next_greatest_letter(["c", "f", "j"], "j"))   # c  (wraps)

c
f
c


## 7. Interview thinking

**Signals:** "first / last occurrence", "insert position", "how many times", "smallest element ≥ x", "largest element ≤ x" (= `upper_bound - 1`).

**Say:** "I want the first index where the condition becomes true. The condition is monotonic, so I binary search with `hi = mid` when it is true, `lo = mid + 1` when false."

**Common mistakes:** `hi = mid - 1` in this template (can skip the answer), `lo <= hi` with `hi = mid` (infinite loop), forgetting that the answer can be `len(nums)`, not checking `nums[first] == target`.

## Quick Revision

- Lower bound: first `≥ x`. Upper bound: first `> x`. Count = upper − lower.
- Template: `lo, hi = 0, n`, `while lo < hi`, true → `hi = mid`, false → `lo = mid + 1`.
- Last position = `upper_bound - 1`. Largest `≤ x` = `upper_bound - 1`.
- Answer can equal `n` → check bounds before indexing.